# 🎬 CineMatch AI: Content-Based Movie Recommendation System
### **Author:** Nandini Mane
### **Project Type:** Machine Learning & Natural Language Processing (NLP)

---

## 📌 Project Overview
This project builds an intelligent Content-Based Movie Recommendation System using the **TMDB 5000 Movie Dataset**.
We extract rich multi-attribute metadata (*plot overview, genres, keywords, top cast, and directors*), process and stem textual tokens using **NLTK**, vectorize text via **Bag-of-Words (CountVectorizer)**, and compute mathematical similarity across 4,800+ movies using **Cosine Similarity**.

## 1. Import Libraries & Load Datasets

In [ ]:
import numpy as np
import pandas as pd
import ast
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from nltk.stem.porter import PorterStemmer
import pickle

# Load TMDB 5000 Datasets
movies = pd.read_csv('tmdb_5000_movies.csv')
credits = pd.read_csv('tmdb_5000_credits.csv')

print(f"Movies shape: {movies.shape}")
print(f"Credits shape: {credits.shape}")

## 2. Dataset Merging & Feature Selection

In [ ]:
# Merge both datasets on movie title
df = movies.merge(credits, on='title')

# Keep relevant features for Content-Based Filtering
df = df[['movie_id', 'title', 'overview', 'genres', 'keywords', 'cast', 'crew', 'vote_average', 'release_date', 'runtime']]
df.dropna(subset=['overview', 'genres', 'keywords', 'cast', 'crew'], inplace=True)
df.head(3)

## 3. Metadata Extraction & Feature Engineering

In [ ]:
def extract_names(obj):
    return [item['name'] for item in ast.literal_eval(obj)]

def extract_top3_cast(obj):
    cast = ast.literal_eval(obj)
    return [item['name'] for item in cast[:3]]

def extract_director(obj):
    for item in ast.literal_eval(obj):
        if item.get('job') == 'Director':
            return item.get('name', 'Unknown')
    return 'Unknown'

df['genres_list'] = df['genres'].apply(extract_names)
df['keywords_list'] = df['keywords'].apply(extract_names)
df['cast_list'] = df['cast'].apply(extract_top3_cast)
df['director'] = df['crew'].apply(extract_director)

df[['title', 'genres_list', 'cast_list', 'director']].head(3)

## 4. NLP Preprocessing & Tag Construction
Remove spaces from entities (e.g., `Sam Worthington` -> `SamWorthington`) to prevent entity collisions, then concatenate all features into a unified text representation.

In [ ]:
clean_genres = df['genres_list'].apply(lambda x: [i.replace(' ', '') for i in x])
clean_keywords = df['keywords_list'].apply(lambda x: [i.replace(' ', '') for i in x])
clean_cast = df['cast_list'].apply(lambda x: [i.replace(' ', '') for i in x])
clean_director = df['director'].apply(lambda x: [x.replace(' ', '')] if x != 'Unknown' else [])
overview_words = df['overview'].apply(lambda x: str(x).split())

df['tags'] = overview_words + clean_genres + clean_keywords + clean_cast + clean_director
df['tags_string'] = df['tags'].apply(lambda x: ' '.join(x).lower())

# Apply Porter Stemming
ps = PorterStemmer()
def stem_text(text):
    return ' '.join([ps.stem(word) for word in text.split()])

df['tags_stemmed'] = df['tags_string'].apply(stem_text)
print('Sample Processed Tag:')
print(df['tags_stemmed'].iloc[0][:200])

## 5. Vector Space Modeling & Cosine Similarity
Transform text into a 5,000-dimensional Bag-of-Words matrix and compute pairwise Cosine Similarity.

In [ ]:
cv = CountVectorizer(max_features=5000, stop_words='english')
vectors = cv.fit_transform(df['tags_stemmed']).toarray()
print(f"Feature matrix shape: {vectors.shape}")

similarity = cosine_similarity(vectors)
print(f"Similarity matrix shape: {similarity.shape}")

## 6. Model Recommendation Function Test

In [ ]:
def recommend_movies(movie_title, top_k=5):
    if movie_title not in df['title'].values:
        return f"Movie '{movie_title}' not found in dataset."
    
    idx = df[df['title'] == movie_title].index[0]
    scores = sorted(list(enumerate(similarity[idx])), key=lambda x: x[1], reverse=True)
    
    print(f"\n🎯 Top {top_k} Recommendations for '{movie_title}':\n" + "-"*50)
    for i, score in scores[1:top_k+1]:
        print(f"🎬 {df.iloc[i]['title']} (Similarity: {round(score*100, 1)}% | Rating: {df.iloc[i]['vote_average']}/10)")

# Test Recommendation Engine
recommend_movies('The Dark Knight', top_k=5)
recommend_movies('Interstellar', top_k=5)